<a href="https://colab.research.google.com/github/swapnilchavan0707/SwapMLDev/blob/main/Primetrade.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

hl_path = '/content/historical_data.csv'
fg_path = '/content/fear_greed_index.csv'

print("Verifying files in your Colab root directory")

if not os.path.exists(hl_path) or not os.path.exists(fg_path):
    print("File path error Current files in your sidebar root are")
    print(os.listdir('/content/'))
    print("Please verify if your filenames match historical_data.csv and fear_greed_index.csv exactly")
else:
    print("Both files successfully located Starting processing pipeline")

    df_hl = pd.read_csv(hl_path)
    df_fg = pd.read_csv(fg_path)

Verifying files in your Colab root directory
Both files successfully located Starting processing pipeline


In [ ]:
print("Columns in Historical Data:")
print(list(df_hl.columns))

print("\nColumns in Fear Greed Index:")
print(list(df_fg.columns))

Columns in Historical Data:
['Account', 'Coin', 'Execution Price', 'Size Tokens', 'Size USD', 'Side', 'Timestamp IST', 'Start Position', 'Direction', 'Closed PnL', 'Transaction Hash', 'Order ID', 'Crossed', 'Fee', 'Trade ID', 'Timestamp']

Columns in Fear Greed Index:
['timestamp', 'value', 'classification', 'date']


In [ ]:
print("Cleaning and synchronising dates")

# 1. Clean Fear & Greed dataset dates using mixed format parsing
df_fg['Clean_Date'] = pd.to_datetime(df_fg['date'], format='mixed').dt.date

# 2. Clean Historical trading dataset dates (Handles raw millisecond numbers or string dates)
if df_hl['Timestamp'].dtype in ['int64', 'float64']:
    df_hl['Clean_Date'] = pd.to_datetime(df_hl['Timestamp'], unit='ms').dt.date
else:
    df_hl['Clean_Date'] = pd.to_datetime(df_hl['Timestamp'], format='mixed').dt.date

# 3. Standardise categorical fields
df_hl['Side'] = df_hl['Side'].astype(str).str.upper()

# 4. Merge the two datasets on the synchronized clean date column
df_merged = pd.merge(df_hl, df_fg, on='Clean_Date', how='inner')
print("Success Data aligned Merged row count")
print(df_merged.shape)

Cleaning and synchronising dates
Success Data aligned Merged row count
(17181, 21)


In [ ]:
print("Engineering analytical trading metrics")

# Calculate metrics using your exact column names
df_merged['is_long'] = np.where(df_merged['Side'].str.contains('BUY|LONG'), 1, 0)
df_merged['is_profitable'] = np.where(df_merged['Closed PnL'] > 0, 1, 0)

# Aggregate daily metrics per trading account
daily_profiles = df_merged.groupby(['Account', 'Clean_Date', 'classification']).agg(
    net_pnl=('Closed PnL', 'sum'),
    total_volume=('Size USD', 'sum'),
    trade_count=('Account', 'count'),
    long_ratio=('is_long', 'mean'),
    win_rate=('is_profitable', 'mean')
).reset_index()
print("Metrics successfully engineered")

Engineering analytical trading metrics
Metrics successfully engineered


In [ ]:
print("INSIGHT 1 Sentiment vs Trader Performance")
sentiment_summary = daily_profiles.groupby('classification').agg(
    avg_pnl=('net_pnl', 'mean'),
    win_rate=('win_rate', 'mean'),
    long_bias=('long_ratio', 'mean'),
    active_traders=('Account', 'nunique')
).reset_index()
print(sentiment_summary.to_string(index=False))

# Distribution chart of net profits across sentiment categories
plt.figure(figsize=(9, 5))
sns.boxplot(data=daily_profiles, x='classification', y='net_pnl', showfliers=False, palette='Set2')
plt.title('Daily Account PnL Distribution across Sentiment Regimes')
plt.savefig('/content/sentiment_pnl_distribution.png', bbox_inches='tight')
plt.close()
print("Plot saved as /content/sentiment_pnl_distribution.png")

INSIGHT 1 Sentiment vs Trader Performance
classification       avg_pnl  win_rate  long_bias  active_traders
 Extreme Greed  59785.971018  0.641679   0.275696               2
          Fear 356131.644903  0.327055   0.583292               7
         Greed  -4169.952057  0.271312   0.457767               7


/tmp/ipykernel_1741/670616272.py:12: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=daily_profiles, x='classification', y='net_pnl', showfliers=False, palette='Set2')


Plot saved as /content/sentiment_pnl_distribution.png


In [ ]:
print("Training Predictive Machine Learning Model")

# Encode classification text strings for modeling
ml_df = pd.get_dummies(daily_profiles, columns=['classification'], drop_first=True)

# Define feature arrays using your specific metrics
features = [col for col in ml_df.columns if 'classification_' in col] + ['total_volume', 'trade_count', 'long_ratio']

# Set target: 1 if trader made net profit today, 0 if flat or negative
ml_df['is_profitable_today'] = np.where(ml_df['net_pnl'] > 0, 1, 0)

X = ml_df[features].fillna(0)
y = ml_df['is_profitable_today']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
print("MODEL PERFORMANCE REPORT")
print(classification_report(y_test, y_pred))

importances = pd.DataFrame({
    'Trading Driver Factor': features,
    'Relative Importance Score': model.feature_importances_
}).sort_values(by='Relative Importance Score', ascending=False)

print("INSIGHT 2 Most Critical Drivers of Trading Profitability")
print(importances.to_string(index=False))

plt.figure(figsize=(9, 5))
sns.barplot(data=importances, x='Relative Importance Score', y='Trading Driver Factor', palette='viridis')
plt.title('Key Drivers of Daily Trader Success')
plt.tight_layout()
plt.savefig('/content/driver_importances.png', bbox_inches='tight')
plt.close()
print("Plot saved as /content/driver_importances.png")
print("Data execution completed successfully")

Training Predictive Machine Learning Model
MODEL PERFORMANCE REPORT
              precision    recall  f1-score   support

           0       0.00      0.00      0.00         2
           1       0.67      1.00      0.80         4

    accuracy                           0.67         6
   macro avg       0.33      0.50      0.40         6
weighted avg       0.44      0.67      0.53         6

INSIGHT 2 Most Critical Drivers of Trading Profitability
Trading Driver Factor  Relative Importance Score
           long_ratio                   0.453780
          trade_count                   0.233171
         total_volume                   0.202976
 classification_Greed                   0.055718
  classification_Fear                   0.054356


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/tmp/ipykernel_1741/915561048.py:33: FutureWarnin

Plot saved as /content/driver_importances.png
Data execution completed successfully
